In [ ]:
#user-fixable errors and interrupt()
#easiest reusable pattern you can use everywhere:

```text
Python
→ LangGraph
→ interrupt()
→ resume()
```

In [ ]:
!pip install langgraph  --quiet

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


In [ ]:
# STATE
class State(TypedDict):
    name: str | None
    result: str | None

# NODE
def ask_name(state: State) -> Command[Literal["ask_name", "__end__"]]:

    # If name missing -> pause graph
    if not state.get("name"):

        print("Pausing graph... waiting for user input")

        user_input = interrupt({
            "message": "Please provide your name"
        })

        print("Graph resumed")

        return Command(
            update={"name": user_input["name"]},
            goto="ask_name"
        )

    # Continue normally
    return Command(
        update={"result": f"Hello {state['name']}"},
        goto=END
    )


# BUILD GRAPH
builder = StateGraph(State)

builder.add_node("ask_name", ask_name)

builder.add_edge(START, "ask_name")


# Checkpointer REQUIRED for interrupt()
app = builder.compile(
    checkpointer=InMemorySaver()
)

# CONFIG

config = {
    "configurable": {
        "thread_id": "demo_1"
    }
}

# FIRST INVOKE
print("\nFIRST INVOKE\n")
result = app.invoke(
    {
        "name": None,
        "result": None
    },
    config=config
)

print(result)


# SECOND INVOKE (RESUME)
print("\nSECOND INVOKE\n")

result = app.invoke(
    Command(
        resume={"name": "Jolly"}
    ),
    config=config
)

print(result)


FIRST INVOKE

Pausing graph... waiting for user input
{'name': None, 'result': None, '__interrupt__': [Interrupt(value={'message': 'Please provide your name'}, id='bf6ca2fdfc05366471c1a176b5b8067f')]}

SECOND INVOKE

Pausing graph... waiting for user input
Graph resumed
{'name': 'Jolly', 'result': 'Hello Jolly'}


In [ ]:
#unexpected error